In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
cd= pd.read_csv("../data/raw/olist_closed_deals_dataset_marketing.csv")

In [4]:
# First rows
cd.head()

,mql_id,seller_id,sdr_id,sr_id,won_date,business_segment,lead_type,lead_behaviour_profile,has_company,has_gtin,average_stock,business_type,declared_product_catalog_size,declared_monthly_revenue
0,5420aad7fec3549a85876ba1c529bd84,2c43fb513632d29b3b58df74816f1b06,a8387c01a09e99ce014107505b92388c,4ef15afb4b2723d8f3d81e51ec7afefe,2018-02-26 19:58:54,pet,online_medium,cat,NaN,NaN,NaN,reseller,NaN,0.0
1,a555fb36b9368110ede0f043dfc3b9a0,bbb7d7893a450660432ea6652310ebb7,09285259593c61296eef10c734121d5b,d3d1e91a157ea7f90548eef82f1955e3,2018-05-08 20:17:59,car_accessories,industry,eagle,NaN,NaN,NaN,reseller,NaN,0.0
2,327174d3648a2d047e8940d7d15204ca,612170e34b97004b3ba37eae81836b4c,b90f87164b5f8c2cfa5c8572834dbe3f,6565aa9ce3178a5caf6171827af3a9ba,2018-06-05 17:27:23,home_appliances,online_big,cat,NaN,NaN,NaN,reseller,NaN,0.0
3,f5fee8f7da74f4887f5bcae2bafb6dd6,21e1781e36faf92725dde4730a88ca0f,56bf83c4bb35763a51c2baab501b4c67,d3d1e91a157ea7f90548eef82f1955e3,2018-01-17 13:51:03,food_drink,online_small,NaN,NaN,NaN,NaN,reseller,NaN,0.0
4,ffe640179b554e295c167a2f6be528e0,ed8cb7b190ceb6067227478e48cf8dde,4b339f9567d060bcea4f5136b9f5949e,d3d1e91a157ea7f90548eef82f1955e3,2018-07-03 20:17:45,home_appliances,industry,wolf,NaN,NaN,NaN,manufacturer,NaN,0.0


In [5]:
# Number of rows and columns
print("Shape:", cd.shape)

# Column names
print("Columns:", cd.columns.tolist())

# Data types and non-null counts
cd.info()

# Statistical summary
cd.describe()

Shape: (842, 14)
Columns: ['mql_id', 'seller_id', 'sdr_id', 'sr_id', 'won_date', 'business_segment', 'lead_type', 'lead_behaviour_profile', 'has_company', 'has_gtin', 'average_stock', 'business_type', 'declared_product_catalog_size', 'declared_monthly_revenue']
<class 'pandas.DataFrame'>
RangeIndex: 842 entries, 0 to 841
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   mql_id                         842 non-null    str    
 1   seller_id                      842 non-null    str    
 2   sdr_id                         842 non-null    str    
 3   sr_id                          842 non-null    str    
 4   won_date                       842 non-null    str    
 5   business_segment               841 non-null    str    
 6   lead_type                      836 non-null    str    
 7   lead_behaviour_profile         665 non-null    str    
 8   has_company                    63 non-n

,declared_product_catalog_size,declared_monthly_revenue
count,69.000000,8.420000e+02
mean,233.028986,7.337768e+04
std,352.380558,1.744799e+06
min,1.000000,0.000000e+00
25%,30.000000,0.000000e+00
50%,100.000000,0.000000e+00
75%,300.000000,0.000000e+00
max,2000.000000,5.000000e+07


In [7]:

# 1. Check missing values
print("Missing values:")
print(cd.isnull().sum())

# 2. Check duplicate rows
print("\nFully duplicated rows:",
      cd.duplicated().sum())

# 3. Check duplicate MQL IDs
print("Duplicate MQL IDs:",
      cd["mql_id"].duplicated().sum())

# 4. Check duplicate seller IDs
print("Duplicate seller IDs:",
      cd["seller_id"].duplicated().sum())

# 5. Check date range
print("\nWon date range:")
print(cd["won_date"].min())
print(cd["won_date"].max())

# 6. Inspect categorical columns
categorical_cols = [
    "business_segment",
    "lead_type",
    "lead_behaviour_profile",
    "has_company",
    "has_gtin",
    "average_stock",
    "business_type"
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(cd[col].value_counts(dropna=False).head(15))

# 7. Investigate declared monthly revenue
print("\nMonthly revenue summary:")
print(cd["declared_monthly_revenue"].describe())

print("\nNumber of zero revenues:",
      (cd["declared_monthly_revenue"] == 0).sum())

# 8. Check product catalogue size
print("\nProduct catalogue size summary:")
print(cd["declared_product_catalog_size"].describe())

# 9. Check negative numerical values
numeric_cols = cd.select_dtypes(include="number").columns

print("\nNegative numerical values:")
print((cd[numeric_cols] < 0).sum())


Missing values:
mql_id                             0
seller_id                          0
sdr_id                             0
sr_id                              0
won_date                           0
business_segment                   1
lead_type                          6
lead_behaviour_profile           177
has_company                      779
has_gtin                         778
average_stock                    776
business_type                     10
declared_product_catalog_size    773
declared_monthly_revenue           0
dtype: int64

Fully duplicated rows: 0
Duplicate MQL IDs: 0
Duplicate seller IDs: 0

Won date range:
2017-12-05 02:00:00
2018-11-14 18:04:19

--- business_segment ---
business_segment
home_decor                         105
health_beauty                       93
car_accessories                     77
household_utilities                 71
construction_tools_house_garden     69
audio_video_electronics             64
computers                           34
pet      

In [9]:

# Create a copy for cleaning
closed_deals_cleaned = cd.copy()

# 1. Convert won_date to datetime
closed_deals_cleaned["won_date"] = pd.to_datetime(
    closed_deals_cleaned["won_date"],
    errors="coerce"
)

# 2. Replace missing values in selected categorical columns
cols_to_fill = [
    "business_segment",
    "lead_type",
    "business_type"
]

for col in cols_to_fill:
    closed_deals_cleaned[col] = (
        closed_deals_cleaned[col].fillna("unknown")
    )

# 3. Preserve other missing values
# No imputation for lead_behaviour_profile, has_company,
# has_gtin, average_stock or declared_product_catalog_size.

# 4. Preserve zero revenue values
# No changes to declared_monthly_revenue.

# 5. Validate the cleaned dataset
print("Original shape:", cd.shape)
print("Cleaned shape:", closed_deals_cleaned.shape)

print("\nUpdated data types:")
print(closed_deals_cleaned.dtypes)

print("\nMissing values:")
print(closed_deals_cleaned.isnull().sum())

print("\nDuplicate MQL IDs:",
      closed_deals_cleaned["mql_id"].duplicated().sum())

print("\nDuplicate seller IDs:",
      closed_deals_cleaned["seller_id"].duplicated().sum())

print("\nCategorical columns after cleaning:")
for col in cols_to_fill:
    print(f"\n{col}:")
    print(closed_deals_cleaned[col].value_counts())

print("\nWon date range:")
print(closed_deals_cleaned["won_date"].min())
print(closed_deals_cleaned["won_date"].max())

print("\nZero revenue values remaining:",
      (closed_deals_cleaned["declared_monthly_revenue"] == 0).sum())

print("\nPreview:")
print(closed_deals_cleaned.head())


Original shape: (842, 14)
Cleaned shape: (842, 14)

Updated data types:
mql_id                                      str
seller_id                                   str
sdr_id                                      str
sr_id                                       str
won_date                         datetime64[us]
business_segment                            str
lead_type                                   str
lead_behaviour_profile                      str
has_company                              object
has_gtin                                 object
average_stock                               str
business_type                               str
declared_product_catalog_size           float64
declared_monthly_revenue                float64
dtype: object

Missing values:
mql_id                             0
seller_id                          0
sdr_id                             0
sr_id                              0
won_date                           0
business_segment                   0
lea

In [10]:
closed_deals_cleaned.to_csv(
    "closed_deals_cleaned.csv",
    index=False
)

print("Closed deals dataset saved successfully!")

Closed deals dataset saved successfully!


Olist Brazilian E-commerce – Marketing Funnel & Seller Acquisition
1. Dataset Overview

The dataset contains 842 records and 14 columns describing qualified leads (MQLs) and acquired sellers in the Brazilian e-commerce market.

Original shape: 842 × 14
Cleaned shape: 842 × 14
Duplicate MQL IDs: 0
Duplicate seller IDs: 0
Won date range: December 2017 – November 2018
Data types were standardized, including datetime conversion for won_date and numerical formats for revenue and product catalog size.

2. Key Findings from Data Cleaning

Missing values: Seller profile attributes such as has_company, has_gtin, average_stock, and declared_product_catalog_size have over 90% missing values.
Business segments: Home decor (105), health and beauty (93), and car accessories (77) are the most represented categories.
Lead types: Online medium leads dominate with 332 records (39.4%).
Business types: Resellers represent 587 records (69.7%), compared with 242 manufacturers (28.7%).
Declared revenue: 797 records (94.7%) have zero declared monthly revenue.
The dataset contains no duplicate identifiers and retains all original observations.

3. Research Questions and Hypotheses

Part B: Marketing Funnel and Seller Acquisition

Research Question 5: How effectively do potential sellers progress through the Olist marketing funnel?

Hypothesis: Conversion performance differs across available lead origins and business segments.

The analysis will investigate lead origins and business segments to identify differences in conversion performance. This requires data covering both converted and non-converted leads.

Research Question 6: Can seller acquisition information be connected to marketplace performance?

The analysis will explore whether acquired seller groups differ in:

Marketplace activity.
Customer review scores.
Delivery performance.
This investigation will only be conducted if a sufficiently complete and reliable relationship between marketing leads and marketplace sellers can be established.

4. Analytical Considerations

Before testing the hypotheses, the following limitations must be addressed:

High missingness in seller profile attributes.
A large proportion of zero declared monthly revenue values.
Verification of the relationship between MQL IDs and seller IDs.
Availability of lost-lead and initial-lead data for funnel conversion calculations.
Availability of reliable seller-level connections to marketplace orders and reviews.
5. Conclusion and Next Steps

The dataset provides a foundation for investigating Olist's seller acquisition process and the characteristics of successfully acquired sellers.

The next step is to examine lead conversion patterns across origins and business segments. If reliable seller-level connections are available, the analysis will be extended to marketplace activity, customer satisfaction, and delivery performance.

The main objective is to understand which types of sellers are successfully acquired and whether their acquisition characteristics are associated with marketplace performance.